In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from datetime import datetime, timedelta
from datetime import datetime, timedelta
55
#padroniza os nomes para facilitar no desenvolvimento ao longo do tempo
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"
catalogo = "catalogo_filmes"

#ajuda a padronizar quando formos ler ou escrever nas tabelas, diminui erros de digitação e facilita manutenção caso seja necessário trocar o nome
bronze_schema = f"{catalogo}.{bronze_schema_name}"
silver_schema = f"{catalogo}.{silver_schema_name}"
gold_schema = f"{catalogo}.{gold_schema_name}"

landing_path = f"/Volumes/{catalogo}/{bronze_schema_name}/landing"



In [0]:
#cria os schemas necessarios caso não existam no catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

Catalog, schemas e volume prontos.


In [0]:
#verificação rápida que garante os arquivos no diretorio correto
#abordagem alternativa que eu poderia usar: se houvesse muitos arquivos, eu poderia listar dinamicamente o
#conteúdo de landing_path e derivar o nome da tabela a partir do nome do arquivo,
#em vez de manter uma lista fixa de arquivos esperados. o ponto negativo é que além de mais lento eu perderia um pouco do controle explicito 
arquivos_esperados = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

for arquivo in arquivos_esperados:
    try:
        dbutils.fs.ls(f"{landing_path}/{arquivo}")  # pegando conteudo dentro da pasta em que os csv estao
    except Exception as e:
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")


In [0]:
mapeamento_arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",          
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

#le os csv e cria a coluna ingestion_datetime
for nome_arquivo, nome_tabela in mapeamento_arquivos_tabelas.items():
    caminho = f"{landing_path}/{nome_arquivo}"
    df = spark.read.csv(caminho, header=True, inferSchema=True)
    df = df.withColumn("ingestion_datetime", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(f"{bronze_schema}.{nome_tabela}")

In [0]:
data_fim_default = datetime.today()
data_inicio_default = data_fim_default - timedelta(days=21)

dbutils.widgets.text("data_inicio", data_inicio_default.strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", data_fim_default.strftime("%m-%d-%Y"))

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")


url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo("
    f"dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url)
response.raise_for_status()  #Verifica se a requisição foi bem-sucedida, caso nao da erro

#pega os dados da resposta(value)
dados_cotacao = response.json()["value"]

#Converte a lista de dicionários em dataframe spark
df_cotacao = spark.createDataFrame(dados_cotacao)
df_cotacao = df_cotacao.withColumn("ingestion_datetime", current_timestamp())

#grava na bronze, usando os metodos pedidos formato delta e modo append
df_cotacao.write.format("delta").mode("append").saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

display(df_cotacao)

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.169,2026-09-14 13:10:08.144425,2026-09-20T16:28:49.496Z
5.1484,2026-09-15 13:09:19.199664,2026-09-20T16:28:49.496Z
5.152,2026-09-16 13:05:30.35873,2026-09-20T16:28:49.496Z
5.1515,2026-09-17 13:03:21.858212,2026-09-20T16:28:49.496Z
5.1569,2026-09-18 13:03:34.742036,2026-09-20T16:28:49.496Z
